In [3]:
import sympy as sp


def var_str(idx, name):
    return f"{name}{idx}"


t = sp.Symbol('t')
q1 = sp.Function('q1')(t)
q2 = sp.Function('q2')(t)
dq1 = sp.diff(q1, t)
dq2 = sp.diff(q2, t)
ddq1 = sp.diff(dq1, t)
ddq2 = sp.diff(dq2, t)

q = [q1, q2]
dq = [dq1, dq2]
ddq = [ddq1, ddq2]

# 参数定义
m1, m2 = sp.symbols('m1 m2', real=True, positive=True)
l1 = sp.symbols('l1', real=True)  # Yaw偏心
l2x, l2z = sp.symbols('l2x l2z', real=True)  # Pitch偏心 (水平和垂直)

# 基座三轴平动加速度/重力分量 (替代固定重力 g)
# 假设接收到基座加速度计的等效引力场矢量为 [gx, gy, gz]^T
gx, gy, gz = sp.symbols('gx gy gz', real=True)

I1xx, I1yy, I1zz = sp.symbols('I1xx I1yy I1zz', real=True, positive=True)
I2xx, I2yy, I2zz = sp.symbols('I2xx I2yy I2zz', real=True, positive=True)

I1 = sp.diag(I1xx, I1yy, I1zz)
I2 = sp.diag(I2xx, I2yy, I2zz)

# Yaw绕Z旋转
R1 = sp.Matrix([
    [sp.cos(q1), -sp.sin(q1), 0],
    [sp.sin(q1),  sp.cos(q1), 0],
    [0,           0,          1]
])
# Pitch绕局部Y旋转
R2_1 = sp.Matrix([
    [sp.cos(q2), 0, sp.sin(q2)],
    [0,          1, 0],
    [-sp.sin(q2), 0, sp.cos(q2)]
])
R2 = R1 * R2_1
print("R1 =", R1)
print("R2 =", R2)

# 质心位置计算
p1 = sp.Matrix([0, 0, l1])  # Yaw轴质心仅在Z轴上有偏移
# Pitch质心设为在自身的X轴有偏移l2x，Z轴有偏移l2z
p2_local = sp.Matrix([l2x, 0, l2z])
p2 = sp.Matrix([0, 0, l1]) + R2 * p2_local

v1 = sp.diff(p1, t)
v2 = sp.diff(p2, t)

# 局部角速度算子
w1_world = sp.Matrix([0, 0, dq1])
w2_world = w1_world + R1 * sp.Matrix([0, dq2, 0])

w1_local = R1.T * w1_world
w2_local = R2.T * w2_world

K1 = 0.5 * m1 * \
    sp.simplify((v1.T * v1)[0]) + 0.5 * \
    sp.simplify((w1_local.T * I1 * w1_local)[0])
K2 = 0.5 * m2 * \
    sp.simplify((v2.T * v2)[0]) + 0.5 * \
    sp.simplify((w2_local.T * I2 * w2_local)[0])
K = sp.simplify(K1 + K2)
print("Kinetic Energy K =", K)

# 将标量 g 替换为 3D 加速度/等效重力场向量
g_vec = sp.Matrix([gx, gy, gz])
# 势能/广义力的等效能量 P = - m * (g_vec \cdot p)
P1 = -m1 * (g_vec.T * p1)[0]
P2 = -m2 * (g_vec.T * p2)[0]
P = sp.simplify(P1 + P2)
print("Potential Energy P =", P)

# M矩阵
M = sp.zeros(2, 2)
for i in range(2):
    for j in range(2):
        M[i, j] = sp.simplify(sp.diff(sp.diff(K, dq[i]), dq[j]))

print("\nM_11 =", M[0, 0])
print("M_12 =", M[0, 1])
print("M_21 =", M[1, 0])
print("M_22 =", M[1, 1])

# C矩阵: 利用Christoffel符号
C = sp.zeros(2, 2)
for k in range(2):
    for j in range(2):
        c_kj = 0
        for i in range(2):
            c_ijk = 0.5 * \
                (sp.diff(M[k, j], q[i]) +
                 sp.diff(M[k, i], q[j]) - sp.diff(M[i, j], q[k]))
            c_kj += c_ijk * dq[i]
        C[k, j] = sp.simplify(c_kj)

print("\nC_11 =", C[0, 0])
print("C_12 =", C[0, 1])
print("C_21 =", C[1, 0])
print("C_22 =", C[1, 1])

# G向量
G = sp.zeros(2, 1)
for i in range(2):
    G[i, 0] = sp.simplify(sp.diff(P, q[i]))

print("\ng_1 =", G[0, 0])
print("g_2 =", G[1, 0])

R1 = Matrix([[cos(q1(t)), -sin(q1(t)), 0], [sin(q1(t)), cos(q1(t)), 0], [0, 0, 1]])
R2 = Matrix([[cos(q1(t))*cos(q2(t)), -sin(q1(t)), sin(q2(t))*cos(q1(t))], [sin(q1(t))*cos(q2(t)), cos(q1(t)), sin(q1(t))*sin(q2(t))], [-sin(q2(t)), 0, cos(q2(t))]])
Kinetic Energy K = 0.5*I1zz*Derivative(q1(t), t)**2 + 0.5*I2xx*sin(q2(t))**2*Derivative(q1(t), t)**2 + 0.5*I2yy*Derivative(q2(t), t)**2 + 0.5*I2zz*cos(q2(t))**2*Derivative(q1(t), t)**2 + 0.25*m2*(l2x**2*cos(2*q2(t))*Derivative(q1(t), t)**2 + l2x**2*Derivative(q1(t), t)**2 + 2*l2x**2*Derivative(q2(t), t)**2 + 2*l2x*l2z*sin(2*q2(t))*Derivative(q1(t), t)**2 - l2z**2*cos(2*q2(t))*Derivative(q1(t), t)**2 + l2z**2*Derivative(q1(t), t)**2 + 2*l2z**2*Derivative(q2(t), t)**2)
Potential Energy P = -gz*l1*m1 - m2*(gx*(l2x*cos(q2(t)) + l2z*sin(q2(t)))*cos(q1(t)) + gy*(l2x*cos(q2(t)) + l2z*sin(q2(t)))*sin(q1(t)) + gz*(l1 - l2x*sin(q2(t)) + l2z*cos(q2(t))))

M_11 = 1.0*I1zz + 1.0*I2xx*sin(q2(t))**2 + 1.0*I2zz*cos(q2(t))**2 + 0.5*m2*(l2x**2*cos(2*q2(t)) + 

In [4]:
# 需要将 dq, ddq 转化为 sp.Matrix 否则会报错
dq_mat = sp.Matrix(dq)
ddq_mat = sp.Matrix(ddq)

# 提取动力学方程等式左边 (tau)
tau_expr = M * ddq_mat + C * dq_mat + G

pi_1, pi_2, pi_3, pi_4, pi_5 = sp.symbols('pi_1 pi_2 pi_3 pi_4 pi_5', real=True)

# 为了做代换，我们需要将原来组合的物理量拆出来
tau_sub = tau_expr.subs({
    I1zz: pi_1,
    I2xx: pi_2,
    I2zz: pi_3 - m2*l2**2,
    I2yy: pi_4 - m2*l2**2
})

# 对于 m2*l2，替换为 pi_5
tau_final = tau_sub.subs({
    m2 * l2: pi_5
})

tau_final = sp.simplify(tau_final)

# 现在构建 Pi 向量
Pi = sp.Matrix([pi_1, pi_2, pi_3, pi_4, pi_5])

# 计算回归矩阵 Y (即 tau_final 关于 Pi 的雅可比矩阵)
Y_matrix = tau_final.jacobian(Pi)
Y_matrix

# print("--- 通过雅可比矩阵求得的回归矩阵 Y (Yaw 轴) ---")
# print("Y_11 (Yaw 对 pi_1):", Y_matrix[0, 0])
# print("Y_12 (Yaw 对 pi_2):", Y_matrix[0, 1])
# print("Y_13 (Yaw 对 pi_3):", Y_matrix[0, 2])
# print("Y_14 (Yaw 对 pi_4):", Y_matrix[0, 3])
# print("Y_15 (Yaw 对 pi_5):", Y_matrix[0, 4])

# print("\n--- 通过雅可比矩阵求得的回归矩阵 Y (Pitch 轴) ---")
# print("Y_21 (Pitch 对 pi_1):", Y_matrix[1, 0])
# print("Y_22 (Pitch 对 pi_2):", Y_matrix[1, 1])
# print("Y_23 (Pitch 对 pi_3):", Y_matrix[1, 2])
# print("Y_24 (Pitch 对 pi_4):", Y_matrix[1, 3])
# print("Y_25 (Pitch 对 pi_5):", Y_matrix[1, 4])

NameError: name 'l2' is not defined

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy.signal import savgol_filter

def identify_parameters(csv_path):
    """
    通过最小二乘法进行云台动力学参数辨识
    增加 Pitch Z 轴偏心重力矩 -g*sin(q2) 项
    """
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"数据文件 {csv_path} 未找到，请准备相应的实验数据文件。")
        return
        
    time = df['time'].values
    q1, q2 = df['q1'].values, df['q2'].values
    dq1, dq2 = df['dq1'].values, df['dq2'].values
    tau1, tau2 = df['tau1'].values, df['tau2'].values
    
    ddq1 = savgol_filter(np.gradient(dq1, time), window_length=11, polyorder=2)
    ddq2 = savgol_filter(np.gradient(dq2, time), window_length=11, polyorder=2)
    
    g = 9.81
    N = len(df)
    
    # 增加的参数为 theta_5 = m2*l2z
    # 后面的摩擦参数顺延
    Y = np.zeros((2 * N, 9))
    tau = np.zeros((2 * N, 1))
    
    # === Yaw ===
    Y[:N, 0] = np.sin(q2)**2 * ddq1 + np.sin(2*q2) * dq1 * dq2
    Y[:N, 1] = np.cos(q2)**2 * ddq1 - np.sin(2*q2) * dq1 * dq2
    Y[:N, 2] = 0
    Y[:N, 3] = 0
    Y[:N, 4] = 0  # Pitch 的 Z向重力项对 Yaw 无影响
    Y[:N, 5] = dq1
    Y[:N, 6] = np.sign(dq1)
    Y[:N, 7] = 0
    Y[:N, 8] = 0
    tau[:N, 0] = tau1
    
    # === Pitch ===
    Y[N:, 0] = -0.5 * np.sin(2*q2) * dq1**2
    Y[N:, 1] = 0.5 * np.sin(2*q2) * dq1**2
    Y[N:, 2] = ddq2
    Y[N:, 3] = -g * np.cos(q2)  # 原本的水平偏心 m2*l2x
    Y[N:, 4] = -g * np.sin(q2)  # 新增的垂直偏心 m2*l2z
    Y[N:, 5] = 0
    Y[N:, 6] = 0
    Y[N:, 7] = dq2
    Y[N:, 8] = np.sign(dq2)
    tau[N:, 0] = tau2
    
    print("Rank of Y:", np.linalg.matrix_rank(Y), " (Expected:", Y.shape[1], ")")
    
    reg = LinearRegression(fit_intercept=False)
    reg.fit(Y, tau)
    theta_eval = reg.coef_[0]
    
    print("--- 辨识参数提取结果 ---")
    print(f"theta_1 (I1zz_com)             : {theta_eval[0]:.6f}")
    print(f"theta_2 (I2xx_com)             : {theta_eval[1]:.6f}")
    print(f"theta_3 (I2yy_com)             : {theta_eval[2]:.6f}")
    print(f"theta_4 (m2*l2x 水平前向偏心)  : {theta_eval[3]:.6f}")
    print(f"theta_5 (m2*l2z 垂直上向偏心)  : {theta_eval[4]:.6f}")
    print(f"theta_6 (fv1)                  : {theta_eval[5]:.6f}")
    print(f"theta_7 (fc1)                  : {theta_eval[6]:.6f}")
    print(f"theta_8 (fv2)                  : {theta_eval[7]:.6f}")
    print(f"theta_9 (fc2)                  : {theta_eval[8]:.6f}")
    
    score = reg.score(Y, tau)
    print(f"\n回归模型拟合优度 R^2 评分: {score:.4f}")
    
    return theta_eval

identify_parameters("ident_data.csv")

Rank of Y: 9  (Expected: 9 )
--- 辨识参数提取结果 ---
theta_1 (I1zz_com)             : 0.113139
theta_2 (I2xx_com)             : 0.127113
theta_3 (I2yy_com)             : 0.027020
theta_4 (m2*l2x 水平前向偏心)  : 0.078564
theta_5 (m2*l2z 垂直上向偏心)  : 0.038247
theta_6 (fv1)                  : 0.001518
theta_7 (fc1)                  : 0.706820
theta_8 (fv2)                  : 0.355941
theta_9 (fc2)                  : 0.037057

回归模型拟合优度 R^2 评分: 0.8769


array([0.11313911, 0.1271133 , 0.02701958, 0.078564  , 0.03824676,
       0.00151766, 0.70682046, 0.3559409 , 0.03705741])